<a href="https://colab.research.google.com/github/Sameer-30/Machine-Learning-Surrogates-/blob/main/Research_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Scope statement
-----------------------------------------
"This surrogate is trained and evaluated to provide rapid demand estimation
WITHIN the studied design space (4 geometries × 3 sub-model typologies ×
15 FEMA P-695 far-field records). Generalization to entirely unseen
ground motion records is acknowledged as a limitation and identified as
future work."

Algorithms
----------
  1. Random Forest Regressor          (sklearn)
  2. XGBoost Regressor                (xgboost)
  3. Support Vector Regression (RBF)  (sklearn)

"""

# ============================================================================
#  0. INSTALL DEPENDENCIES
# ============================================================================
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib seaborn

# ============================================================================
#  1. IMPORTS
# ============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import (KFold, ShuffleSplit, RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import (r2_score, mean_squared_error,
                             mean_absolute_error,
                             mean_absolute_percentage_error)

import xgboost as xgb

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[Note] SHAP not installed — SHAP plots will be skipped.")

warnings.filterwarnings('ignore')

# ============================================================================
#  2. USER CONFIGURATION
# ============================================================================
DATA_DIR   = '/content'
OUTPUT_DIR = '/content/ML_Results_Final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBMODELS = ['BareFrame', 'Infill', 'Openstory']

TARGET = 'ln_roof_disp_envelope'
TARGET_DISPLAY = 'ln(EDP)'


DROP_COLS = [
    'sample_id', 'model_id', 'submodel', 'geometry',
    'record_id', 'record_short', 'component',
    'roof_disp_max', 'ln_roof_disp_max',
    'roof_disp_envelope', 'ln_roof_disp_envelope',
    'roof_disp_srss', 'damage_state',
]


HEATMAP_FEATURES = [
    'PGA',          # peak ground acceleration
    'PGV',          # peak ground velocity
    'PGD',          # peak ground displacement
    'Sa_T1',        # spectral acceleration at fundamental period
    'Ia',           # Arias intensity
    'D5_95',        # significant duration
    'CAV',          # cumulative absolute velocity
    'Tp',           # predominant period
    'PGV_PGA',      # pulse indicator (PGV/PGA ratio)
]


# ---- Multi-seed evaluation ----
SEEDS         = list(range(10))    # 10 seeds for robust mean ± std
TEST_FRAC     = 0.20
CV_SPLITS     = 5
N_ITER_SEARCH = 30


plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         11,
    'axes.labelsize':    13,
    'axes.titlesize':    14,
    'legend.fontsize':   10,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'figure.dpi':        110,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
})

# ============================================================================
#  3. HELPERS
# ============================================================================

def load_and_clean(submodel_name):
    """Load CSV, drop NaN target rows, drop zero-variance feature columns."""
    path = os.path.join(DATA_DIR, f'ML_Dataset_{submodel_name}.csv')
    df = pd.read_csv(path)
    n0 = len(df)

    df = df.dropna(subset=[TARGET]).reset_index(drop=True)

    drop = [c for c in DROP_COLS if c in df.columns]
    X = df.drop(columns=drop)
    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        X = X.drop(columns=non_numeric)


    nunique = X.nunique()
    zero_var = nunique[nunique <= 1].index.tolist()
    X = X.drop(columns=zero_var)

    mask = X.notna().all(axis=1)
    X = X.loc[mask].reset_index(drop=True)
    y = df.loc[mask, TARGET].reset_index(drop=True)

    print(f"  Loaded {submodel_name}: {n0} → {len(X)} rows after cleaning")
    print(f"    Final feature count : {X.shape[1]}")
    return X, y


def metrics_dict(y_true, y_pred):
    return {
        'R2':   r2_score(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':  mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred),
    }


# ============================================================================
#  4. DESCRIPTIVE STATISTICS
# ============================================================================

def make_descriptive_stats_table(X, y, sm_name):
    """
    Generate a descriptive statistics table (count, mean, SD, min, max, quartiles)
    formatted like the reference paper's Table 7.
    Includes both features and the target.
    """
    df = X.copy()
    df[TARGET_DISPLAY] = y.values
    desc = df.describe().T  # transpose so rows = parameters
    desc = desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
    desc.columns = ['count', 'mean', 'S.D.', 'min', '25%', '50%', '75%', 'max']
    desc = desc.round(4)
    desc.index.name = 'Parameter'
    out_path = os.path.join(OUTPUT_DIR, f'DescriptiveStats_{sm_name}.csv')
    desc.to_csv(out_path)
    print(f"    Descriptive stats table → {out_path}")
    return desc


# ============================================================================
#  5. CORRELATION MATRIX HEATMAP
# ============================================================================

def make_correlation_heatmap(X, y, sm_name):
    """
    Pearson correlation matrix between selected ground-motion IM features
    and the target. Uses HEATMAP_FEATURES to keep the figure compact and
    paper-ready (mirrors the reference paper's 10×10 layout).
    """

    if HEATMAP_FEATURES is not None:
        keep = [c for c in HEATMAP_FEATURES if c in X.columns]
        if not keep:
            print(f"    [Warn] No HEATMAP_FEATURES present in {sm_name}; "
                  f"falling back to all numeric features.")
            df = X.copy()
        else:
            df = X[keep].copy()
    else:
        df = X.copy()

    df[TARGET_DISPLAY] = y.values
    corr = df.corr(method='pearson')
    n = len(corr)


    fig_size = max(7, 0.6 * n + 1.8)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

    sns.heatmap(
        corr,
        annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        square=True,
        cbar_kws={'shrink': 0.8, 'label': ''},
        annot_kws={'size': 10},
        linewidths=0.5, linecolor='white',
        ax=ax,
    )
    ax.set_xlabel('Features')
    ax.set_ylabel('Features')

    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, f'CorrelationMatrix_{sm_name}.png')
    plt.savefig(out_path)
    plt.show()
    print(f"    Correlation heatmap → {out_path}  ({n}×{n})")
    return corr


# ============================================================================
#  6. MODEL FACTORIES  +  HYPER-PARAMETER GRIDS
# ============================================================================

def build_models():
    models = {}

    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    rf_params = {
        'n_estimators':      [200, 300, 500, 800],
        'max_depth':         [None, 4, 6, 8, 12],
        'min_samples_split': [2, 4, 6, 10],
        'min_samples_leaf':  [1, 2, 4, 6],
        'max_features':      ['sqrt', 'log2', 0.5, 0.7],
    }
    models['RandomForest'] = (rf, rf_params)

    xg = xgb.XGBRegressor(
        random_state=42, n_jobs=-1,
        objective='reg:squarederror', verbosity=0,
    )
    xg_params = {
        'n_estimators':      [100, 200, 400, 600],
        'max_depth':         [3, 4, 5, 6],
        'learning_rate':     [0.01, 0.03, 0.05, 0.08],
        'subsample':         [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree':  [0.6, 0.8, 1.0],
        'reg_alpha':         [0, 0.01, 0.1, 1.0],
        'reg_lambda':        [0.5, 1.0, 2.0, 5.0],
    }
    models['XGBoost'] = (xg, xg_params)

    svr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svr',    SVR(kernel='rbf')),
    ])
    svr_params = {
        'svr__C':       [0.1, 1, 5, 10, 50, 100],
        'svr__gamma':   ['scale', 'auto', 0.01, 0.05, 0.1, 0.5],
        'svr__epsilon': [0.001, 0.01, 0.05, 0.1],
    }
    models['SVR_RBF'] = (svr_pipe, svr_params)

    return models


# ============================================================================
#  7. PER-SUBMODEL TRAINING DRIVER
# ============================================================================

def train_one_submodel(sm_name):
    print("\n" + "=" * 78)
    print(f"  TRAINING — {sm_name}")
    print("=" * 78)

    X, y = load_and_clean(sm_name)
    if len(X) < 30:
        print(f"  [SKIP] Insufficient data ({len(X)} rows)")
        return None

    # ------------------------------------------------------------------
    # STEP A — Descriptive statistics + correlation heatmap
    # ------------------------------------------------------------------
    print("\n  [A] Descriptive statistics & correlation analysis")
    make_descriptive_stats_table(X, y, sm_name)
    make_correlation_heatmap(X, y, sm_name)

    # ------------------------------------------------------------------
    # STEP B — Hyper-parameter tuning (standard 5-fold CV)
    # ------------------------------------------------------------------
    print("\n  [B] Hyper-parameter tuning  (5-fold standard CV)")
    cv_inner = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)
    best_params = {}
    for name, (estimator, param_dist) in build_models().items():
        n_iter = min(N_ITER_SEARCH,
                     int(np.prod([len(v) for v in param_dist.values()])))
        search = RandomizedSearchCV(
            estimator, param_distributions=param_dist,
            n_iter=n_iter, cv=cv_inner, scoring='r2',
            n_jobs=-1, random_state=42, verbose=0,
        )
        search.fit(X, y)
        best_params[name] = search.best_params_
        print(f"    {name:<14s}  best CV R² = {search.best_score_:+.4f}")

    # ------------------------------------------------------------------
    # STEP C — Multi-seed evaluation (10 random 80/20 splits)
    # ------------------------------------------------------------------
    print(f"\n  [C] Multi-seed evaluation ({len(SEEDS)} seeds, ShuffleSplit)")
    per_seed_records = {name: [] for name in best_params}
    fitted_best = {}
    best_seed_score = {name: -np.inf for name in best_params}
    holdout_for_plot = {}

    for seed in SEEDS:
        ss = ShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=seed)
        train_idx, test_idx = next(ss.split(X, y))
        Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
        ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

        for name, (estimator, _) in build_models().items():
            model = clone(estimator)
            model.set_params(**best_params[name])
            model.fit(Xtr, ytr)
            ypred = model.predict(Xte)
            tm = metrics_dict(yte, ypred)
            per_seed_records[name].append(tm)

            if tm['R2'] > best_seed_score[name]:
                best_seed_score[name] = tm['R2']
                fitted_best[name] = model
                holdout_for_plot[name] = (
                    Xte.copy(), yte.copy(), ypred.copy()
                )

    rows = []
    for name, records in per_seed_records.items():
        df_seeds = pd.DataFrame(records)
        rows.append({
            'SubModel':        sm_name,
            'Algorithm':       name,
            'R2_mean':         df_seeds['R2'].mean(),
            'R2_std':          df_seeds['R2'].std(),
            'R2_min':          df_seeds['R2'].min(),
            'R2_max':          df_seeds['R2'].max(),
            'RMSE_mean':       df_seeds['RMSE'].mean(),
            'RMSE_std':        df_seeds['RMSE'].std(),
            'MAE_mean':        df_seeds['MAE'].mean(),
            'MAE_std':         df_seeds['MAE'].std(),
            'MAPE_mean':       df_seeds['MAPE'].mean(),
            'N_seeds':         len(records),
        })
        print(f"    {name:<14s}  R² = {df_seeds['R2'].mean():+.4f} "
              f"± {df_seeds['R2'].std():.4f}  |  "
              f"RMSE = {df_seeds['RMSE'].mean():.4f} ± "
              f"{df_seeds['RMSE'].std():.4f}")

    df_master = pd.DataFrame(rows)

    # ------------------------------------------------------------------
    # STEP D — Predicted vs Actual (best-seed visualization)
    # ------------------------------------------------------------------
    print(f"\n  [D] Generating Predicted vs Actual plots")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax, name in zip(axes, ['RandomForest', 'XGBoost', 'SVR_RBF']):
        Xte, yte, ypred = holdout_for_plot[name]
        r2 = r2_score(yte, ypred)
        rmse = np.sqrt(mean_squared_error(yte, ypred))
        ax.scatter(yte, ypred, alpha=0.75, s=50,
                   edgecolor='k', linewidth=0.5, color='steelblue')
        lo = float(min(yte.min(), ypred.min()))
        hi = float(max(yte.max(), ypred.max()))
        margin = (hi - lo) * 0.05
        ax.plot([lo - margin, hi + margin], [lo - margin, hi + margin],
                'r--', lw=1.4, label='1:1 line')
        ax.set_xlabel(f'Actual {TARGET_DISPLAY}')
        ax.set_ylabel(f'Predicted {TARGET_DISPLAY}')
        ax.set_title(f'{name}\n$R^2$ = {r2:.3f}  |  RMSE = {rmse:.3f}')
        ax.legend(loc='upper left')
    fig.suptitle(f'Predicted vs Actual — {sm_name}',
                 y=1.02, fontsize=14, fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f'PredVsActual_{sm_name}.png'))
    plt.show()

    # ------------------------------------------------------------------
    # STEP E — Feature importance (RF + XGBoost)
    # ------------------------------------------------------------------
    print(f"  [E] Generating Feature Importance plots")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, name in zip(axes, ['RandomForest', 'XGBoost']):
        m = fitted_best[name]
        importances = m.feature_importances_
        n_top = min(10, len(importances))
        idx = np.argsort(importances)[::-1][:n_top]
        ax.barh(range(n_top), importances[idx][::-1],
                color='steelblue', edgecolor='k', linewidth=0.5)
        ax.set_yticks(range(n_top))
        ax.set_yticklabels(np.array(X.columns)[idx][::-1])
        ax.set_xlabel('Importance')
        # ax.set_title(f'{name}')
    fig.suptitle(f'Feature Importance (top {n_top}) — {sm_name}',
                 y=1.02, fontsize=14, fontweight='bold')
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f'FeatureImportance_{sm_name}.png'))
    plt.show()

    # ------------------------------------------------------------------
    # STEP F — SHAP summary for best XGBoost model
    # ------------------------------------------------------------------
    if SHAP_AVAILABLE:
        print(f"  [F] Generating SHAP plot (XGBoost)")
        try:
            explainer = shap.TreeExplainer(fitted_best['XGBoost'])
            Xte_b, _, _ = holdout_for_plot['XGBoost']
            shap_vals = explainer.shap_values(Xte_b)
            plt.figure()
            shap.summary_plot(shap_vals, Xte_b, show=False, max_display=10)
            # plt.title(f'SHAP Summary — XGBoost ({sm_name})',
            #           fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f'SHAP_{sm_name}.png'),
                        dpi=300, bbox_inches='tight')
            plt.show()
        except Exception as e:
            print(f"    [Warn] SHAP failed: {e}")

    return df_master


# ============================================================================
#  8. MAIN LOOP
# ============================================================================

print("=" * 78)
print("  ML PIPELINE  (FINAL)  —  Seismic Demand Surrogate for RC Frames")
print("  Target          : ln(roof_disp_envelope)")
print("  Algorithms      : RandomForest, XGBoost, SVR (RBF)")
print(f"  CV strategy     : standard 5-fold (within-design-space evaluation)")
print(f"  Test evaluation : ShuffleSplit × {len(SEEDS)} seeds (mean ± std)")
print("=" * 78)

all_rows = []
for sm in SUBMODELS:
    df_sm = train_one_submodel(sm)
    if df_sm is not None:
        all_rows.append(df_sm)

if not all_rows:
    raise RuntimeError("No sub-models produced results — check DATA_DIR paths.")

master = pd.concat(all_rows, ignore_index=True)
master_path = os.path.join(OUTPUT_DIR, 'ML_Results_Final.csv')
master.to_csv(master_path, index=False)

# ============================================================================
#  9. MASTER COMPARISON TABLE  +  SUMMARY BAR CHART
# ============================================================================

print("\n" + "=" * 78)
print("  PAPER RESULTS TABLE  (mean ± std across 10 seeds, standard CV)")
print("=" * 78)


paper_table = master.copy()
paper_table['R²']   = paper_table.apply(
    lambda r: f"{r['R2_mean']:.4f} ± {r['R2_std']:.4f}", axis=1)
paper_table['RMSE'] = paper_table.apply(
    lambda r: f"{r['RMSE_mean']:.4f} ± {r['RMSE_std']:.4f}", axis=1)
paper_table['MAE']  = paper_table.apply(
    lambda r: f"{r['MAE_mean']:.4f} ± {r['MAE_std']:.4f}", axis=1)
paper_display = paper_table[['SubModel', 'Algorithm', 'R²', 'RMSE', 'MAE']]
print(paper_display.to_string(index=False))

paper_display_path = os.path.join(OUTPUT_DIR, 'PaperTable_ML_Results.csv')
paper_display.to_csv(paper_display_path, index=False)
print(f"\n  Paper-formatted table : {paper_display_path}")
print(f"  Full results CSV      : {master_path}")


fig, ax = plt.subplots(figsize=(11, 5.5))
algorithms = ['RandomForest', 'XGBoost', 'SVR_RBF']
sm_list = [sm for sm in SUBMODELS if sm in master['SubModel'].unique()]
x = np.arange(len(algorithms))
width = 0.25
colors = ['#1f77b4', '#2ca02c', '#d62728']

for i, sm in enumerate(sm_list):
    sub = master[master['SubModel'] == sm].set_index('Algorithm').loc[algorithms]
    ax.bar(x + i*width - width, sub['R2_mean'], width,
           yerr=sub['R2_std'], capsize=4,
           color=colors[i], edgecolor='k', linewidth=0.6, label=sm)

ax.set_xticks(x)
ax.set_xticklabels(algorithms)
ax.set_ylabel('Test $R^2$  (mean ± std, n = 10 seeds)')
ax.set_title('Algorithm Performance Across Sub-models')
ax.axhline(0, color='k', lw=0.7)
ax.legend(title='Sub-model', frameon=True, loc='lower right')
ax.set_ylim(min(0, master['R2_mean'].min() - 0.1), 1.0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'AlgorithmComparison_R2.png'))
plt.show()

combined_desc = []
for sm in sm_list:
    desc_path = os.path.join(OUTPUT_DIR, f'DescriptiveStats_{sm}.csv')
    if os.path.exists(desc_path):
        d = pd.read_csv(desc_path)
        d.insert(0, 'SubModel', sm)
        combined_desc.append(d)
if combined_desc:
    pd.concat(combined_desc, ignore_index=True).to_csv(
        os.path.join(OUTPUT_DIR, 'DescriptiveStats_All.csv'), index=False)

print("\n" + "=" * 78)
print(f"  ALL OUTPUTS SAVED IN: {OUTPUT_DIR}")
print("=" * 78)
